# Spaceship Titanic — Preprocesamiento de datos

**Materia:** Inteligencia Artificial Avanzada
**Equipo:** _(completar)_
**Integrantes:** _(completar)_
**Fecha:** _(completar)_

---

Este notebook cubre el pipeline de preparación de datos del reto
[Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic):
carga, EDA, selección de variables, manejo de faltantes, codificación,
transformación, escalado y verificación final.

Las celdas marcadas como **Interpretación** contienen el razonamiento del equipo
detrás de cada decisión; las celdas de código sólo generan la evidencia.

## 0. Configuración del entorno

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")  # oculta warnings para no ensuciar la salida

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # semilla fija para reproducibilidad

# Opciones de despliegue de pandas/matplotlib/seaborn
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)

# Los CSV viven en <repo>/data/spaceship-titanic/.
# Buscamos en varias rutas para que funcione tanto si ejecutas el notebook
# desde SpaceShip_Titanic/ como desde la raiz del repositorio.
CANDIDATAS = [
    Path("../data/spaceship-titanic"),
    Path("data/spaceship-titanic"),
    Path("data"),
]
DATA_DIR = next(
    (c for c in CANDIDATAS if (c / "train.csv").exists()),
    CANDIDATAS[0],
)  # primera ruta candidata que exista; si ninguna existe, usa la primera como default
print("DATA_DIR:", DATA_DIR.resolve())

print("pandas", pd.__version__, "| numpy", np.__version__)  # versiones para reproducibilidad

## 1. Carga y comprensión del dataset

In [ ]:
def obtener_datos(data_dir: Path = DATA_DIR) -> tuple[Path, Path]:
    """Devuelve las rutas de train.csv y test.csv.

    Intenta usar la copia local en data/; si no existe, descarga desde Kaggle
    con kagglehub (requiere credenciales y haber aceptado las reglas).
    """
    train_local, test_local = data_dir / "train.csv", data_dir / "test.csv"
    if train_local.exists() and test_local.exists():
        print("Usando copia local en", data_dir.resolve())
        return train_local, test_local

    # No hay copia local: se descarga el dataset de la competencia desde Kaggle
    import shutil

    import kagglehub

    ruta = Path(kagglehub.competition_download("spaceship-titanic"))
    for nombre in ("train.csv", "test.csv"):
        shutil.copy(ruta / nombre, data_dir / nombre)
    print("Descargado desde Kaggle y copiado a", data_dir.resolve())
    return train_local, test_local


ruta_train, ruta_test = obtener_datos()
train = pd.read_csv(ruta_train)  # dataset de entrenamiento (incluye el target)
test = pd.read_csv(ruta_test)  # dataset de prueba (sin target, para predecir)

print("train:", train.shape, "| test:", test.shape)

In [ ]:
train.head()  # primeras filas para inspeccion visual rapida

In [ ]:
train.info()  # tipos de dato y conteo de no-nulos por columna

In [ ]:
# Confirma tamanios y que la unica diferencia de columnas entre train y test sea el target
print("train shape:", train.shape)
print("test shape:", test.shape)

cols_train = set(train.columns)
cols_test = set(test.columns)
print("Columnas en train pero no en test:", cols_train - cols_test)
print("Columnas en test pero no en train:", cols_test - cols_train)

In [ ]:
train.describe()  # estadisticos descriptivos de las columnas numericas

In [ ]:
train.describe(include="object")  # conteo, cardinalidad y valor mas frecuente de las columnas categoricas

In [ ]:
train["Transported"].value_counts(normalize=True)  # proporcion de cada clase del target

### Interpretación — variable target y tipo de problema

Respondan usando la evidencia de las celdas de arriba:

1. ¿Cuál es la variable target y qué tipo de dato tiene?
La variable target es TRansported, tipo bool.

2. ¿Qué tipo de problema es (clasificación/regresión, binaria/multiclase)?
Clasificación binaria.

3. Según el `value_counts(normalize=True)`, ¿las clases están balanceadas o hay desbalance? ¿Qué tan severo?
Sería un balance con un porcentaje de 50.4%/49.6%, por lo que practicamente se encuentra balanceado.

4. ¿Por qué `test.csv` tiene una columna menos que `train.csv`?
La columna que tiene menos justo es la variable target que es Transported, debido a que test.csv nos permitirá generar predicciones nuevas. Kaggle la oculta para poder evaluar las predicciones contra el valor real que tienen ellos.

## 2. Análisis exploratorio (EDA)

### 2.1 Faltantes: informativos vs. reales

In [13]:
# Porcentaje de valores faltantes por columna, solo las que tienen al menos un faltante
faltantes_pct = train.isna().mean().sort_values(ascending=False) * 100
faltantes_pct = faltantes_pct[faltantes_pct > 0]
faltantes_pct

CryoSleep       2.496261
ShoppingMall    2.392730
VIP             2.335212
HomePlanet      2.312205
Name            2.300702
Cabin           2.289198
VRDeck          2.162660
Spa             2.105142
FoodCourt       2.105142
Destination     2.093639
RoomService     2.082135
Age             2.059128
dtype: float64

In [14]:
# Estadisticos de gasto para pasajeros con CryoSleep=True: si la logica del dominio se cumple, deberia ser todo 0
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
train.loc[train["CryoSleep"] == True, spend_cols].describe()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,2969.0,2967.0,2941.0,2972.0,2975.0
mean,0.0,0.0,0.0,0.0,0.0
std,0.0,0.0,0.0,0.0,0.0
min,0.0,0.0,0.0,0.0,0.0
25%,0.0,0.0,0.0,0.0,0.0
50%,0.0,0.0,0.0,0.0,0.0
75%,0.0,0.0,0.0,0.0,0.0
max,0.0,0.0,0.0,0.0,0.0


In [15]:
# Gasto total (suma de las 5 columnas) agrupado por CryoSleep, para comparar el patron entre True y False
train.assign(TotalSpend=train[spend_cols].sum(axis=1)).groupby("CryoSleep")["TotalSpend"].describe()

,count,mean,std,min,25%,50%,75%,max
CryoSleep,,,,,,,,
False,5439.0,2248.299687,3245.061489,0.0,746.0,1019.0,2416.0,35987.0
True,3037.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


#### Interpretación — clasificación de faltantes

Para **cada** columna con faltantes (la tabla de `faltantes_pct`), clasifíquenla como *informativo* o *real*, con evidencia:

1. ¿Qué porcentaje de faltantes tiene cada columna? ¿Son similares entre sí o hay alguna muy distinta al resto? (un patrón muy uniforme entre columnas sugiere faltantes inyectados al azar, no informativos).
2. Con la evidencia de las dos celdas anteriores: cuando `CryoSleep=True`, ¿cuánto gastan los pasajeros en promedio? ¿Y cuando es `False`? ¿Qué les dice esto sobre imputar los gastos faltantes según el valor de `CryoSleep`?
3. Para el resto de las columnas (`HomePlanet`, `Cabin`, `Destination`, `VIP`, `Name`, `Age`), ¿encuentran alguna relación lógica similar con otra variable, o su ausencia parece simplemente aleatoria? Justifiquen columna por columna, no en bloque.

_(completar)_

### 2.2 Variables numéricas: distribución, skewness y outliers

### 2.3 Variables categóricas: cardinalidad y desbalance

### 2.4 Relación de cada variable con el target

## 3. Selección y descarte de variables

## 4. Manejo de faltantes

## 5. Codificación de variables categóricas

## 6. Transformación de variables numéricas

## 7. Escalado

## 8. Verificación final

### 8.1 Tabla resumen de decisiones

### 8.2 Conclusión

---

## Nota de uso de IA

_(completar)_